# SODIndoorLoc — preparación y clustering autocontenidos


## 2. Importaciones y rutas


In [1]:
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]
print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


## 3. Lectura y preprocesado RSSI

Se normalizan los nombres de columnas, se detectan automáticamente WAP/MAC
y el escalador se ajusta solo con la partición de entrenamiento.


In [2]:
def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def find_file_case_insensitive(filename: str, directories: Sequence[Path]) -> Path:
    """Busca un nombre sin depender de mayusculas/minusculas."""
    checked: List[str] = []
    target = filename.casefold()
    for directory in directories:
        directory = Path(directory)
        checked.append(str(directory / filename))
        if not directory.exists():
            continue
        direct = directory / filename
        if direct.exists():
            return direct.resolve()
        for child in directory.iterdir():
            if child.is_file() and child.name.casefold() == target:
                return child.resolve()
    raise FileNotFoundError(
        f"No se encontro {filename}. Rutas comprobadas:\n- " + "\n- ".join(checked)
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().upper() for c in out.columns]
    return out


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


## 4. Coordenadas y división de posiciones


In [3]:
def add_sod_metric_targets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["TARGET_X_M"] = pd.to_numeric(out["ECOORD"], errors="raise").astype(float)
    out["TARGET_Y_M"] = pd.to_numeric(out["NCOORD"], errors="raise").astype(float)
    out["METRIC_CRS"] = "SOD_LOCAL_METRES"
    return out


def position_key(df: pd.DataFrame, columns: Sequence[str]) -> pd.Series:
    return df[list(columns)].astype(str).agg("|".join, axis=1)


def split_positions_stratified(
    df: pd.DataFrame,
    position_columns: Sequence[str],
    client_column: str,
    val_size: float = 0.15,
    seed: int = SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Separa posiciones completas para impedir coordenadas repetidas entre train/val."""
    work = df.copy()
    work["_POSITION_KEY"] = position_key(work, position_columns)
    positions = (
        work.groupby("_POSITION_KEY", as_index=False)
        .agg(**{client_column: (client_column, lambda s: s.mode().iloc[0])})
    )
    stratify = positions[client_column]
    if stratify.value_counts().min() < 2:
        stratify = None
    train_keys, val_keys = train_test_split(
        positions["_POSITION_KEY"],
        test_size=val_size,
        random_state=seed,
        stratify=stratify,
    )
    train_out = work[work["_POSITION_KEY"].isin(set(train_keys))].drop(columns="_POSITION_KEY")
    val_out = work[work["_POSITION_KEY"].isin(set(val_keys))].drop(columns="_POSITION_KEY")
    return train_out.reset_index(drop=True), val_out.reset_index(drop=True)


def reconstruct_sod_official_raw(
    leaky_train: pd.DataFrame,
    leaky_test: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    """Recupera el split oficial SOD por multiplicidad de puntos (30 train/10 test)."""
    full = pd.concat([normalize_columns(leaky_train), normalize_columns(leaky_test)], ignore_index=True)
    pos_cols = [c for c in ["BUILDINGID", "FLOORID", "ECOORD", "NCOORD"] if c in full.columns]
    if not {"ECOORD", "NCOORD"}.issubset(pos_cols):
        raise ValueError("SOD necesita ECOORD y NCOORD para reconstruir el split oficial.")
    full["_POSITION_KEY"] = position_key(full, pos_cols)
    counts = full.groupby("_POSITION_KEY").size()
    unexpected = counts[~counts.isin([10, 30])]
    if len(unexpected):
        raise ValueError(
            "No se puede reconstruir de forma segura el split oficial: "
            f"{len(unexpected)} posiciones no tienen 10 o 30 capturas. "
            "Proporciona Training_HCXY_All.csv y Testing_HCXY_All.csv."
        )
    train_keys = set(counts[counts == 30].index)
    test_keys = set(counts[counts == 10].index)
    train = full[full["_POSITION_KEY"].isin(train_keys)].drop(columns="_POSITION_KEY").reset_index(drop=True)
    test = full[full["_POSITION_KEY"].isin(test_keys)].drop(columns="_POSITION_KEY").reset_index(drop=True)
    diagnostics = {
        "method": "reconstructed_from_30_vs_10_scans_per_position",
        "n_train_rows": int(len(train)),
        "n_test_rows": int(len(test)),
        "n_train_positions": int(len(train_keys)),
        "n_test_positions": int(len(test_keys)),
    }
    return train, test, diagnostics


## 5. Guardado y router RSSI→clúster

Los identificadores de fila permiten unir cada partición con sus rutas.
KMeans se ajusta con las posiciones de train; un Extra Trees aprende a
reproducir esas zonas desde RSSI para validación y test.


### 5.1. Guardado de las tres particiones


In [4]:
def _assign_row_ids(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    out.insert(0, "ROW_ID", [f"{prefix}_{i:07d}" for i in range(len(out))])
    return out


def save_base_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Dict[str, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    split_frames = {
        "train": _assign_row_ids(train, f"{prefix}_train"),
        "val": _assign_row_ids(val, f"{prefix}_val"),
        "test": _assign_row_ids(test, f"{prefix}_test"),
    }
    paths: Dict[str, Path] = {}
    for split, frame in split_frames.items():
        path = output_dir / f"{prefix}_{split}.csv"
        frame.to_csv(path, index=False)
        paths[split] = path
    return paths


def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


### 5.2. Entrenamiento del router y auditoría


In [5]:
def fit_rssi_routing(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    rssi_columns: Sequence[str],
    k_values: Sequence[int],
    seed: int = SEED,
    n_estimators: int = 200,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Define zonas con XY de train y aprende RSSI -> zona para val/test.

    CLUSTER_ORACLE solo se conserva para diagnostico. La columna CLUSTER que
    consumen los modelos es predicha por RSSI en validacion y test.
    """
    pre = RSSIPreprocessor(use_mask=True).fit(train, rssi_columns)
    x_train = pre.transform(train)
    x_val = pre.transform(val)
    x_test = pre.transform(test)
    coords_train = train[TARGET_COLUMNS].to_numpy(dtype=float)
    unique_coords = np.unique(coords_train, axis=0)

    route_parts: List[pd.DataFrame] = []
    diagnostics: List[Dict[str, object]] = []
    for k in sorted(set(int(v) for v in k_values)):
        if k < 2 or k > len(unique_coords):
            continue
        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=20)
        kmeans.fit(unique_coords)
        oracle = {
            "train": kmeans.predict(train[TARGET_COLUMNS].to_numpy(dtype=float)),
            "val": kmeans.predict(val[TARGET_COLUMNS].to_numpy(dtype=float)),
            "test": kmeans.predict(test[TARGET_COLUMNS].to_numpy(dtype=float)),
        }
        gate = ExtraTreesClassifier(
            n_estimators=n_estimators,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed + k,
            n_jobs=-1,
        )
        gate.fit(x_train, oracle["train"])
        predicted = {
            "train": oracle["train"],
            "val": gate.predict(x_val),
            "test": gate.predict(x_test),
        }
        probabilities = {
            "train": np.ones(len(train), dtype=float),
            "val": np.max(gate.predict_proba(x_val), axis=1),
            "test": np.max(gate.predict_proba(x_test), axis=1),
        }
        for split, frame in [("train", train), ("val", val), ("test", test)]:
            route_parts.append(
                pd.DataFrame(
                    {
                        "ROW_ID": frame["ROW_ID"].astype(str).to_numpy(),
                        "SPLIT": split,
                        "N_CLUSTERS": k,
                        "CLUSTER": predicted[split].astype(int),
                        "CLUSTER_ORACLE": oracle[split].astype(int),
                        "GATE_CONFIDENCE": probabilities[split].astype(float),
                    }
                )
            )
        diagnostics.append(
            {
                "N_CLUSTERS": k,
                "VAL_GATE_ACCURACY": float(accuracy_score(oracle["val"], predicted["val"])),
                "TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY": float(
                    accuracy_score(oracle["test"], predicted["test"])
                ),
                "VAL_MEAN_CONFIDENCE": float(np.mean(probabilities["val"])),
                "TEST_MEAN_CONFIDENCE": float(np.mean(probabilities["test"])),
                "N_TRAIN_POSITIONS": int(len(unique_coords)),
            }
        )
    if not route_parts:
        raise ValueError("No se genero ninguna configuracion de clustering.")
    return pd.concat(route_parts, ignore_index=True), pd.DataFrame(diagnostics)


def save_routing(
    routes: pd.DataFrame,
    diagnostics: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Tuple[Path, Path]:
    output_dir = Path(output_dir)
    route_path = output_dir / f"{prefix}_routes.csv"
    diagnostics_path = output_dir / f"{prefix}_routing_diagnostics.csv"
    routes.to_csv(route_path, index=False)
    diagnostics.to_csv(diagnostics_path, index=False)
    return route_path, diagnostics_path


def validate_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    position_columns: Sequence[str],
) -> Dict[str, object]:
    ids = [set(frame["ROW_ID"].astype(str)) for frame in [train, val, test]]
    if ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]:
        raise AssertionError("ROW_ID se solapa entre particiones.")
    pos = [set(position_key(frame, position_columns)) for frame in [train, val, test]]
    return {
        "rows": {"train": len(train), "val": len(val), "test": len(test)},
        "positions": {"train": len(pos[0]), "val": len(pos[1]), "test": len(pos[2])},
        "position_overlap": {
            "train_val": len(pos[0] & pos[1]),
            "train_test": len(pos[0] & pos[2]),
            "val_test": len(pos[1] & pos[2]),
        },
    }


## 6. Preparación completa de SOD


In [6]:
def prepare_sod_datasets(
    data_directories: Sequence[Path],
    output_dir: Path,
    k_values: Sequence[int] = tuple(range(2, 9)),
    val_size: float = 0.15,
    seed: int = SEED,
    gate_estimators: int = 200,
) -> Dict[str, object]:
    """Prepara SOD RAW y AVG con el split oficial y rutas predichas por RSSI."""
    dirs = [Path(p) for p in data_directories]
    reconstruction_info: Dict[str, object]
    try:
        raw_train_path = find_file_case_insensitive("Training_HCXY_All.csv", dirs)
        raw_test_path = find_file_case_insensitive("Testing_HCXY_All.csv", dirs)
        raw_official_train = normalize_columns(pd.read_csv(raw_train_path))
        raw_official_test = normalize_columns(pd.read_csv(raw_test_path))
        reconstruction_info = {"method": "official_raw_files"}
    except FileNotFoundError:
        leaky_train_path = find_file_case_insensitive("HCXY_Train_85.csv", dirs)
        leaky_test_path = find_file_case_insensitive("HCXY_Test_15.csv", dirs)
        raw_official_train, raw_official_test, reconstruction_info = reconstruct_sod_official_raw(
            pd.read_csv(leaky_train_path), pd.read_csv(leaky_test_path)
        )

    avg_train_path = find_file_case_insensitive("Training_HCXY_All_Avg.csv", dirs)
    avg_test_path = find_file_case_insensitive("Testing_HCXY_All_Avg.csv", dirs)
    avg_official_train = normalize_columns(pd.read_csv(avg_train_path))
    avg_official_test = normalize_columns(pd.read_csv(avg_test_path))

    results: Dict[str, object] = {"raw_split_source": reconstruction_info, "variants": {}}
    for variant, official_train, official_test in [
        ("sod_raw", raw_official_train, raw_official_test),
        ("sod_avg", avg_official_train, avg_official_test),
    ]:
        required = ["ECOORD", "NCOORD", "PHONEID"]
        missing = [c for c in required if c not in official_train.columns]
        if missing:
            raise ValueError(f"Faltan columnas SOD en {variant}: {missing}")
        pos_cols = [c for c in ["BUILDINGID", "FLOORID", "ECOORD", "NCOORD"] if c in official_train.columns]
        train, val = split_positions_stratified(
            official_train,
            position_columns=pos_cols,
            client_column="PHONEID",
            val_size=val_size,
            seed=seed,
        )
        test = official_test.copy().reset_index(drop=True)
        for frame in [train, val, test]:
            frame["CLIENT_ID"] = frame["PHONEID"].astype(str)
        train, val, test = map(add_sod_metric_targets, [train, val, test])
        paths = save_base_splits(train, val, test, output_dir, variant)
        saved = read_base_splits(output_dir, variant)
        rssi = detect_rssi_columns(saved["train"])
        routes, route_diag = fit_rssi_routing(
            saved["train"], saved["val"], saved["test"], rssi, k_values, seed, gate_estimators
        )
        route_paths = save_routing(routes, route_diag, output_dir, variant)
        diagnostics = validate_splits(saved["train"], saved["val"], saved["test"], pos_cols)
        diagnostics["clients"] = {
            split: sorted(frame["CLIENT_ID"].astype(str).unique().tolist())
            for split, frame in saved.items()
        }
        results["variants"][variant] = {
            "files": {**{k: str(v) for k, v in paths.items()}, "routes": str(route_paths[0])},
            "diagnostics": diagnostics,
            "routing": route_diag.to_dict("records"),
        }
    output_dir = Path(output_dir)
    with (output_dir / "sod_preparation_report.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, ensure_ascii=False)
    return results


## 7. Configuración y ejecución


In [7]:
# Coloca los cuatro CSV SOD en SOD/, Dataset/ o en la carpeta desde la que
# arrancaste Jupyter. Para RAW también se admiten los dos CSV 85/15 antiguos:
# HCXY_Train_85.csv y HCXY_Test_15.csv.
DATA_DIRS = [
    Path.cwd(),
    ROOT / "SOD",
    ROOT / "Dataset",
    ROOT.parent,
    ROOT.parent / "Dataset",
    Path("/mnt/data"),
]
OUTPUT_DIR = ROOT / "prepared" / "SOD"

K_VALUES = list(range(2, 11))
VALIDATION_SIZE = 0.15
SEED = 42
GATE_TREES = 200

report = prepare_sod_datasets(
    data_directories=DATA_DIRS,
    output_dir=OUTPUT_DIR,
    k_values=K_VALUES,
    val_size=VALIDATION_SIZE,
    seed=SEED,
    gate_estimators=GATE_TREES,
)
print("Datos preparados en:", OUTPUT_DIR)


Datos preparados en: /home/coder/Indoor/Notebooks/prepared/SOD


## 8. Auditoría final


In [8]:
# Auditoría obligatoria: los solapamientos train/test deben ser cero.
for variant, info in report["variants"].items():
    print("\n===", variant, "===")
    display(pd.DataFrame([info["diagnostics"]["rows"]], index=["filas"]))
    display(pd.DataFrame([info["diagnostics"]["positions"]], index=["posiciones"]))
    display(pd.DataFrame([info["diagnostics"]["position_overlap"]], index=["solapamiento"]))
    display(pd.DataFrame(info["routing"]))

assert report["variants"]["sod_raw"]["diagnostics"]["position_overlap"]["train_test"] == 0
assert report["variants"]["sod_avg"]["diagnostics"]["position_overlap"]["train_test"] == 0
print("\nOK: no hay coordenadas compartidas entre train oficial y test oficial.")



=== sod_raw ===


,train,val,test
filas,9660,1710,860


,train,val,test
posiciones,322,57,86


,train_val,train_test,val_test
solapamiento,0,0,0


,N_CLUSTERS,VAL_GATE_ACCURACY,TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY,VAL_MEAN_CONFIDENCE,TEST_MEAN_CONFIDENCE,N_TRAIN_POSITIONS
0,2,0.984211,0.989535,0.977658,0.974333,322
1,3,0.984211,0.974419,0.952980,0.941322,322
2,4,0.971345,0.974419,0.933707,0.925513,322
3,5,0.955556,0.932558,0.922313,0.900595,322
4,6,0.939766,0.954651,0.905459,0.875592,322
5,7,0.949123,0.930233,0.873691,0.854440,322
6,8,0.933333,0.965116,0.868678,0.841730,322
7,9,0.952047,0.945349,0.840811,0.833432,322
8,10,0.943275,0.920930,0.829729,0.831327,322


...

,train,val,test
filas,322,57,86


,train,val,test
posiciones,322,57,86


,train_val,train_test,val_test
solapamiento,0,0,0


,N_CLUSTERS,VAL_GATE_ACCURACY,TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY,VAL_MEAN_CONFIDENCE,TEST_MEAN_CONFIDENCE,N_TRAIN_POSITIONS
0,2,0.982456,0.988372,0.974020,0.961912,322
1,3,0.982456,0.988372,0.945177,0.928464,322
2,4,0.929825,0.976744,0.914939,0.893482,322
3,5,0.947368,0.953488,0.907215,0.859983,322
4,6,0.964912,0.941860,0.876116,0.826846,322
5,7,0.947368,0.941860,0.828044,0.807029,322
6,8,0.964912,0.941860,0.840886,0.784345,322
7,9,0.964912,0.918605,0.809187,0.761197,322
8,10,0.964912,0.918605,0.784111,0.752409,322



OK: no hay coordenadas compartidas entre train oficial y test oficial.
